# 09a · Datación PCA-MM12 sin `torch` (iteración 3b, tarea D3)

**Objetivo.** Llevar a Python, sin depender de `torch`, la datación de regímenes por PCA del
notebook 09 y del Rmd (bloques `b8-datacion`, `b8-estabilidad-cortes`, `bB6-datacion` y
`bB6-estabilidad`), en las dos vías (D1 y D2), y ampliarla con la grilla de maxcl y criterios
que pide el plan de la iteración 3b (`asesoria/2026-10-21_plan-iteracion-3b.md`, tarea D3).

**Método (el mismo que el Rmd).** PCA de la matriz de correlaciones con la regla de signo de
Rcamiz; scores PC1–PC3 suavizados con una media móvil **causal** de 12 meses; partición óptima
de Fisher (`src/particion.py`, tarea D2) de los scores estandarizados, con segmentos de al menos
12 meses (`n_min = 12`). El número de regímenes de la especificación principal sale de la
**regla K** de D12 (`decisiones.qmd`): el menor $k$ cuya reducción relativa de la suma de
cuadrados al pasar a $k+1$ es menor que 10 % y, si ninguno lo es, la caída máxima de esa
reducción. El CH se informa en sus dos variantes y no se usa como selector (D11).

**Pasos.**
1. Reproducir `09_regimenes_pca.csv` de las dos vías con maxcl = 10 (fechas, $k$, duración,
   componente de crédito y fase). La columna `episodios_contenidos` no se reproduce aquí: lee
   fechas de episodios escritas a mano, que la tarea D9 sustituirá por el archivo único de
   episodios (D7); no se copian fechas de episodios en este notebook.
2. Estabilidad de los cortes ante la incertidumbre de las cargas: 200 réplicas bootstrap por
   bloques de 24 meses, con los **mismos inicios de bloque** que sortea el Rmd
   (`R_bootstrap_indices_L24.csv`, semilla 2026 del Rmd).
3. Grilla maxcl ∈ {10, 15, 20} × criterio {codo por caída máxima, CH con $k$, CH con $k-1$,
   regla K}, con la columna `k_en_tope`.

**Semillas.** Este notebook no sortea nada: las réplicas son las del Rmd. `SEMILLA = None`.

**Salidas** (vía en niveles en `data/results/niveles/`, transformada en `data/results/`):
`09a_regimenes_pca.csv`, `09a_estabilidad_cortes.csv` y `09a_datacion_criterios.csv`.

**Orden de ejecución.** El Rmd escribe los índices bootstrap y sus propios CSV
(`R_estabilidad_cortes.csv`, `R_datacion_criterios.csv`); si existen, este notebook los compara
con los suyos y falla si difieren. La tabla de paridad del Rmd hace la comparación inversa.

In [1]:
# ---- Configuración --------------------------------------------------------------
import sys
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path.cwd().resolve()
if not (RAIZ / '_quarto.yml').exists():
    RAIZ = RAIZ.parent
assert (RAIZ / '_quarto.yml').exists(), 'No se encontró la raíz del repositorio'
sys.path.insert(0, str(RAIZ))
from src import particion as pt
from src.datos import cargar_vias, CREDITOS
from src.remuestreo import leer_inicios_bootstrap, bootstrap_pca

SEMILLA = None            # sin sorteos propios: réplicas del Rmd
L_MM = 12                 # media móvil causal (meses)
N_MIN = 12                # longitud mínima de un régimen (meses)
UMBRAL = 0.10             # regla K, paso 1 (D12)
D = 3                     # componentes de la datación (D9: d = 3)
MAXCL_GRILLA = (10, 15, 20)
B_CORTES, L_BOOT, TOL_MESES = 200, 24, 3
DIR_T = RAIZ / 'data' / 'results'
DIR_N = DIR_T / 'niveles'

X, Xt = cargar_vias(RAIZ)
VIAS = {'niveles': (X, DIR_N), 'transformadas': (Xt, DIR_T)}
for via, (M, _) in VIAS.items():
    print(via, M.shape, M.index[0], '→', M.index[-1])

niveles (228, 17) 2007-01 → 2025-12
transformadas (227, 17) 2007-02 → 2025-12


## 1. Datación con maxcl = 10: reproducción de `09_regimenes_pca.csv`

Fases (regla del 09): **expansión** si el promedio del componente de crédito (media de los
tres créditos estandarizados en toda la muestra) es mayor que 0,25, **contracción** si es menor
que −0,25 y **transición** si está entre ambos o si el régimen dura menos de 18 meses.

In [2]:
def suavizar(M):
    # Scores PC1..PC3 con la regla de Rcamiz y media móvil causal de 12 meses (sin los NaN iniciales)
    val, V, S = pt.scores_pca_rcamiz(M.to_numpy())
    Sm = np.column_stack([pt.media_movil_causal(S[:, k], L_MM) for k in range(D)])[L_MM - 1:]
    return val, V, Sm, list(M.index[L_MM - 1:])


def fases(M, cortes, fechas_seg, umbral=0.25, dur_min=18):
    Z = pt.estandarizar_r(M.to_numpy())
    pos = [list(M.columns).index(c) for c in CREDITOS]
    comp = (np.sum(Z[:, pos].astype(np.longdouble), axis=1) / len(pos)).astype(float)   # rowMeans
    fechas = list(M.index)
    bordes = [0] + list(cortes) + [len(fechas_seg)]
    filas = []
    for a, b in zip(bordes[:-1], bordes[1:]):
        ini, fin = fechas_seg[a], fechas_seg[b - 1]
        sel = np.array([ini <= f <= fin for f in fechas])
        m, dur = float(np.mean(comp[sel])), int(sel.sum())
        fase = 'Transición' if dur < dur_min or abs(m) <= umbral else ('Expansión' if m > 0 else 'Contracción')
        filas.append({'inicio': ini, 'fin': fin, 'meses': dur, 'componente_credito_medio': round(m, 2), 'fase': fase})
    return pd.DataFrame(filas)


datacion = {}
for via, (M, d) in VIAS.items():
    val, V, Sm, fz = suavizar(M)
    dat = pt.datar(Sm, k_max=10, n_min=N_MIN, umbral=UMBRAL)
    cortes = dat['res'][dat['k_codo']]['cortes']
    reg = fases(M, cortes, fz)
    reg.to_csv(d / '09a_regimenes_pca.csv', index=False)
    ref = pd.read_csv(d / '09_regimenes_pca.csv')
    # Fechas, k, duración y fase exactos; componente de crédito con la tolerancia del Rmd (2 decimales)
    assert len(ref) == len(reg) == dat['k_codo'], (via, len(ref), len(reg))
    for c in ['inicio', 'fin', 'meses', 'fase']:
        assert (ref[c].astype(str).to_numpy() == reg[c].astype(str).to_numpy()).all(), (via, c)
    assert np.max(np.abs(ref.componente_credito_medio - reg.componente_credito_medio)) < 0.0051
    datacion[via] = {'val': val, 'V': V, 'Sm': Sm, 'fz': fz, 'dat': dat, 'cortes': cortes}
    print(f"{via}: k = {dat['k_codo']} (regla K, paso «{dat['regla_codo']}»); reproduce 09_regimenes_pca.csv")
    display(reg)

niveles: k = 6 (regla K, paso «caida»); reproduce 09_regimenes_pca.csv


,inicio,fin,meses,componente_credito_medio,fase
0,2007-12,2008-11,12,-1.45,Transición
1,2008-12,2010-02,15,-1.30,Transición
2,2010-03,2014-03,49,-0.79,Contracción
3,2014-04,2020-05,74,0.17,Transición
4,2020-06,2022-10,29,0.97,Expansión
5,2022-11,2025-12,38,1.39,Expansión


transformadas: k = 8 (regla K, paso «umbral»); reproduce 09_regimenes_pca.csv


,inicio,fin,meses,componente_credito_medio,fase
0,2008-01,2009-05,17,0.66,Transición
1,2009-06,2010-06,13,0.22,Transición
2,2010-07,2013-02,32,0.57,Expansión
3,2013-03,2016-03,37,-0.07,Transición
4,2016-04,2020-02,47,-0.24,Transición
5,2020-03,2021-11,21,-0.49,Contracción
6,2021-12,2023-10,23,-0.31,Contracción
7,2023-11,2025-12,26,-0.52,Contracción


## 2. Estabilidad de los cortes (port de `b8-estabilidad-cortes`)

Para cada réplica $b$ (las 200 primeras de `R_bootstrap_indices_L24.csv`), las cargas PC1–PC4
se emparejan con las originales como en el bootstrap del 05 (`src/remuestreo.bootstrap_pca`);
los scores se recalculan con $V_b = C_b / \sqrt{\lambda_b}$ sobre las series estandarizadas de la
muestra completa, se suavizan y se parten en el mismo número de regímenes. Se cuenta la
proporción de réplicas con un corte a ±3 meses de cada corte original.

*Supuesto:* el bootstrap por bloques supone dependencia débil; en niveles no se cumple, y la
cifra mide estabilidad descriptiva de las fechas, no incertidumbre muestral.

In [3]:
estabilidad = {}
for via, (M, d) in VIAS.items():
    r = datacion[via]
    cargas_ref = r['V'] * np.sqrt(r['val'])
    inicios = leer_inicios_bootstrap(d / 'R_bootstrap_indices_L24.csv')[:B_CORTES]
    bt = bootstrap_pca(M.to_numpy(), cargas_ref, inicios, L_BOOT, K=4)
    Z = pt.estandarizar_r(M.to_numpy())
    k_ref = r['dat']['k_codo']
    frec, _ = pt.estabilidad_cortes(Z, bt['cargas'], bt['valores_emparejados'], r['cortes'], k_ref,
                                    ventana=L_MM, n_min=N_MIN, tolerancia=TOL_MESES, d=D)
    tab = pd.DataFrame({'corte': pt.fechas_corte(r['cortes'], r['fz']), 'frecuencia': frec, 'k_ref': k_ref,
                        'replicas': B_CORTES, 'L': L_BOOT, 'tolerancia_meses': TOL_MESES})
    tab.to_csv(d / '09a_estabilidad_cortes.csv', index=False)
    estabilidad[via] = tab
    f_r = d / 'R_estabilidad_cortes.csv'
    if f_r.exists():
        ref = pd.read_csv(f_r)
        assert list(ref.corte) == list(tab.corte), via
        dif = float(np.max(np.abs(ref.frecuencia.to_numpy() - tab.frecuencia.to_numpy())))
        assert dif < 1e-12, (via, dif)
        print(f'{via}: coincide con R_estabilidad_cortes.csv (dif. máx. {dif:.1e})')
    else:
        print(f'{via}: aún no existe {f_r.name} (ejecutar el Rmd)')
    display(tab)

niveles: coincide con R_estabilidad_cortes.csv (dif. máx. 0.0e+00)


,corte,frecuencia,k_ref,replicas,L,tolerancia_meses
0,2008-12,0.850,6,200,24,3
1,2010-03,0.980,6,200,24,3
2,2014-04,0.350,6,200,24,3
3,2020-06,0.955,6,200,24,3
4,2022-11,1.000,6,200,24,3


transformadas: coincide con R_estabilidad_cortes.csv (dif. máx. 0.0e+00)


,corte,frecuencia,k_ref,replicas,L,tolerancia_meses
0,2009-06,1.000,8,200,24,3
1,2010-07,1.000,8,200,24,3
2,2013-03,0.930,8,200,24,3
3,2016-04,0.860,8,200,24,3
4,2020-03,0.930,8,200,24,3
5,2021-12,0.875,8,200,24,3
6,2023-11,1.000,8,200,24,3


## 3. Grilla maxcl × criterio

`k_max_efectivo` es el mayor $k$ factible con segmentos de al menos 12 meses (18 con
maxcl = 20 en las dos vías). `k_en_tope` indica si el criterio eligió ese máximo.

In [4]:
grillas = {}
for via, (M, d) in VIAS.items():
    r = datacion[via]
    g = pt.grilla_datacion(r['Sm'], r['fz'], maxcls=MAXCL_GRILLA, n_min=N_MIN, umbral=UMBRAL)
    # No regresión: con maxcl = 10, la regla K reproduce la datación del paso 1
    fila = g[(g.maxcl == 10) & (g.criterio == 'regla K (D12)')].iloc[0]
    assert fila.k == r['dat']['k_codo'] and fila.cortes == ' '.join(pt.fechas_corte(r['cortes'], r['fz']))
    g.to_csv(d / '09a_datacion_criterios.csv', index=False)
    grillas[via] = g
    f_r = d / 'R_datacion_criterios.csv'
    if f_r.exists():
        ref = pd.read_csv(f_r, keep_default_na=False)
        clave = lambda t: [f'{a} {b} {c} {k} {str(e).upper()} {rk} {ct}' for a, b, c, k, e, rk, ct in
                           zip(t.maxcl, t.k_max_efectivo, t.criterio, t.k, t.k_en_tope, t.regla_k, t.cortes)]
        g2 = pd.read_csv(d / '09a_datacion_criterios.csv', keep_default_na=False)
        n_dif = sum(a != b for a, b in zip(clave(ref), clave(g2)))
        assert len(ref) == len(g2) and n_dif == 0, (via, n_dif)
        print(f'{via}: coincide con R_datacion_criterios.csv ({len(g2)} filas)')
    else:
        print(f'{via}: aún no existe {f_r.name} (ejecutar el Rmd)')
    display(g)

niveles: coincide con R_datacion_criterios.csv (12 filas)


,maxcl,k_max_efectivo,criterio,k,k_en_tope,regla_k,cortes
0,10,10,codo (caída máxima),6,False,caida,2008-12 2010-03 2014-04 2020-06 2022-11
1,10,10,Calinski-Harabász k gl (aede),10,True,,2008-12 2010-02 2011-05 2014-01 2015-09 2017-0...
2,10,10,Calinski-Harabász k-1 gl (habitual),10,True,,2008-12 2010-02 2011-05 2014-01 2015-09 2017-0...
3,10,10,regla K (D12),6,False,caida,2008-12 2010-03 2014-04 2020-06 2022-11
4,15,15,codo (caída máxima),6,False,caida,2008-12 2010-03 2014-04 2020-06 2022-11
5,15,15,Calinski-Harabász k gl (aede),12,False,,2008-12 2010-02 2011-05 2014-01 2015-09 2017-0...
6,15,15,Calinski-Harabász k-1 gl (habitual),12,False,,2008-12 2010-02 2011-05 2014-01 2015-09 2017-0...
7,15,15,regla K (D12),11,False,umbral,2008-12 2010-02 2011-05 2014-01 2015-09 2017-0...
8,20,18,codo (caída máxima),6,False,caida,2008-12 2010-03 2014-04 2020-06 2022-11
9,20,18,Calinski-Harabász k gl (aede),12,False,,2008-12 2010-02 2011-05 2014-01 2015-09 2017-0...


transformadas: coincide con R_datacion_criterios.csv (12 filas)


,maxcl,k_max_efectivo,criterio,k,k_en_tope,regla_k,cortes
0,10,10,codo (caída máxima),6,False,caida,2009-06 2010-07 2013-05 2022-02 2023-11
1,10,10,Calinski-Harabász k gl (aede),9,False,,2009-06 2010-07 2013-03 2016-04 2020-04 2021-0...
2,10,10,Calinski-Harabász k-1 gl (habitual),8,False,,2009-06 2010-07 2013-03 2016-04 2020-03 2021-1...
3,10,10,regla K (D12),8,False,umbral,2009-06 2010-07 2013-03 2016-04 2020-03 2021-1...
4,15,15,codo (caída máxima),6,False,caida,2009-06 2010-07 2013-05 2022-02 2023-11
5,15,15,Calinski-Harabász k gl (aede),9,False,,2009-06 2010-07 2013-03 2016-04 2020-04 2021-0...
6,15,15,Calinski-Harabász k-1 gl (habitual),8,False,,2009-06 2010-07 2013-03 2016-04 2020-03 2021-1...
7,15,15,regla K (D12),8,False,umbral,2009-06 2010-07 2013-03 2016-04 2020-03 2021-1...
8,20,18,codo (caída máxima),17,False,caida,2009-06 2010-06 2011-06 2012-06 2013-06 2014-0...
9,20,18,Calinski-Harabász k gl (aede),9,False,,2009-06 2010-07 2013-03 2016-04 2020-04 2021-0...


**Lectura.** Este notebook no interpreta los regímenes: deja la grilla y la estabilidad de los
cortes como insumo de la fase 1, donde cada $k^*$ se lee contra el nulo de estructura constante
(D7 del plan, D10 y D11 de `decisiones.qmd`) y nunca contra los episodios (regla 6).